In [11]:
from pathlib import Path
import pandas as pd
import gseapy as gp
from openai import OpenAI

import os

os.environ["OPENAI_API_KEY"] = "your_api_key_here"

from openai import OpenAI

client = OpenAI()

class PathwayAnalysisAgent:

    def __init__(self,
                 results_dir,
                 organism="mus musculus"):

        self.results_dir = Path(results_dir)
        self.organism = organism

    ####################################################
    # Load DE table
    ####################################################

    def load_de(self, comparison):

        file = self.results_dir / f"DE_{comparison}.csv"

        return pd.read_csv(file)

    ####################################################
    # Run enrichment
    ####################################################

    def enrichment(self, df):

        sig = df[
            (df.padj < 0.05)
            &
            (abs(df.log2FC) > 0.5)
        ]

        enr = gp.enrichr(
            gene_list=sig.gene.tolist(),
            gene_sets=[
                "GO_Biological_Process_2023",
                "KEGG_2021_Mouse",
                "Reactome_2022",
                "MSigDB_Hallmark_2020"
            ],
            organism=self.organism,
            outdir=None
        )

        return enr.results

    ####################################################
    # AI interpretation
    ####################################################

    def interpret(self,
                  comparison,
                  enrichment_df):

        top = enrichment_df.head(20)

        prompt = f"""
You are an expert in

- Alzheimer's disease
- Spatial transcriptomics
- Mouse hippocampus
- Aging biology

Comparison:

{comparison}

Top enriched pathways:

{top.to_markdown(index=False)}

Write

1. Summary

2. Biological interpretation

3. Alzheimer's relevance

4. Aging relevance

5. Important genes

6. Novel findings

7. Suggested validation experiments

8. References that should be investigated.

"""

        response = client.responses.create(
            model="gpt-5",
            input=prompt
        )

        return response.output_text

    ####################################################
    # Complete workflow
    ####################################################

    def analyze(self,
                comparison):

        df = self.load_de(comparison)

        pathways = self.enrichment(df)

        report = self.interpret(
            comparison,
            pathways
        )

        return pathways, report

In [12]:
RESULTS_DIR = "results_hippocampus"

"""
comparisons = [
    "AD_vs_Control",
    "Aged_vs_Young",
    "YAD_vs_YC",
    "AAD_vs_AC",
    "AC_vs_YC",
    "AAD_vs_YAD"
]
"""

comparisons = [
    "AAD_vs_AC_Aged_AD_vs_Aged_Control",
    "AAD_VS_YAD_Aged_AD_vs_Young_AD",
    "AC_vs_YC_Aged_Control_vs_Young_Control_(aging_effect)",
    "AD_vs_Control_AII_AD_vs_AII_Control",
    "Aged_vs_Young_AII_Aged_vs_AII_Young",
    "YAD_vs_YC_Young_AD_vs_Young_Control"
]

agent = PathwayAnalysisAgent(
    results_dir=RESULTS_DIR
)

for comp in comparisons:

    pathways, report = agent.analyze(comp)

    pathways.to_csv(
        f"{RESULTS_DIR}/{comp}_pathways.csv",
        index=False
    )

    with open(
        f"{RESULTS_DIR}/{comp}_AI_Report.md",
        "w"
    ) as f:

        f.write(report)

2026-07-13 15:57:18,130 [WARNING] Input library not found: KEGG_2021_Mouse. Skip


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: your_api*****here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [13]:
# =============================================================================
# Pathway Analysis Agent — Spatial Transcriptomics DE Results
# =============================================================================
# Designed to run AFTER the Unified Spatial Transcriptomics DE Pipeline.
#
# Workflow:
#   0. Configuration & Imports
#   1. Load DE Results
#   2. Prepare Gene Set Libraries (Mouse-native MSigDB + Enrichr)
#   3. Over-Representation Analysis (ORA) — significant genes
#   4. GSEA Prerank — full ranked gene lists
#   5. Per-Comparison Pathway Figures (dot plots, bar plots, enrichment plots)
#   6. Cross-Comparison Pathway Heatmap
#   7. Leading-Edge Gene Analysis
#   8. Summary Report & Export
# =============================================================================

# =============================================================================
# 0. Configuration & Imports
# =============================================================================

import gseapy as gp
from gseapy import dotplot, barplot
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
import seaborn as sns
import os
import warnings
import json
from collections import defaultdict
from datetime import datetime

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ─── User Configuration ──────────────────────────────────────────────────────

# Thresholds from your DE pipeline
GENE_PCT = 0.01
MIN_SAMPLES = 4
gene_pct_str = f"{GENE_PCT*100:g}pct"

# Directories — must match your DE pipeline output
RESULTS_DIR = f"results_hippocampus_{gene_pct_str}_min{MIN_SAMPLES}"
PATHWAY_DIR = f"pathway_analysis_hippocampus_{gene_pct_str}_min{MIN_SAMPLES}"
PATHWAY_FIG_DIR = os.path.join(PATHWAY_DIR, "figures")

os.makedirs(PATHWAY_DIR, exist_ok=True)
os.makedirs(PATHWAY_FIG_DIR, exist_ok=True)

# ─── DE significance thresholds ──────────────────────────────────────────────
PADJ_THRESH = 0.05           # adjusted p-value cutoff for ORA gene lists
LOG2FC_THRESH = 0.5          # |log2FC| cutoff for ORA gene lists

# ─── GSEA prerank settings ───────────────────────────────────────────────────
GSEA_PERMUTATIONS = 1000     # number of permutations (increase for publication)
GSEA_MIN_SIZE = 10           # minimum gene set size
GSEA_MAX_SIZE = 500          # maximum gene set size
GSEA_SEED = 42               # reproducibility seed
GSEA_THREADS = 4             # parallel threads

# ─── Ranking metric for prerank ──────────────────────────────────────────────
# Options: "signed_neg_log10p", "log2FC", "t_statistic"
# "signed_neg_log10p" = sign(log2FC) × -log10(pval) — recommended for limma-voom
RANK_METRIC = "signed_neg_log10p"

# ─── ORA settings ────────────────────────────────────────────────────────────
ORA_TOP_TERMS = 20           # max terms to display in plots
ORA_CUTOFF = 0.05            # adjusted p-value cutoff for reporting

# ─── Gene set libraries to query ─────────────────────────────────────────────
# These are fetched from Enrichr (Mouse organism) or MSigDB (Mouse collections).
# The agent will try each and skip unavailable ones gracefully.

# Enrichr mouse libraries (case-sensitive names from gp.get_library_name("Mouse"))
ENRICHR_MOUSE_LIBS = [
    "GO_Biological_Process_2023",
    "GO_Molecular_Function_2023",
    "GO_Cellular_Component_2023",
    "KEGG_2019_Mouse",
    "WikiPathway_2023_Mouse",
    "Reactome_2022",
    "BioPlanet_2019",
]

# MSigDB mouse collections (fetched via gp.Msigdb)
# MH = mouse hallmark, M2 = curated, M5 = ontology
MSIGDB_MOUSE_COLLECTIONS = {
    "Hallmark": "mh.all",
    "C2_Curated": "m2.all",
    "C5_GO_BP": "m5.go.bp",
}
MSIGDB_VERSION = "2024.1.Mm"  # update to latest available

# ─── Comparisons — must match your DE pipeline ──────────────────────────────
COMPARISONS = [
    {"name": "AD_vs_Control",  "desc": "All AD vs All Control"},
    {"name": "Aged_vs_Young",  "desc": "All Aged vs All Young"},
    {"name": "YAD_vs_YC",      "desc": "Young AD vs Young Control"},
    {"name": "AAD_vs_AC",      "desc": "Aged AD vs Aged Control"},
    {"name": "AC_vs_YC",       "desc": "Aged Control vs Young Control"},
    {"name": "AAD_vs_YAD",     "desc": "Aged AD vs Young AD"},
]

# ─── Plot styling ────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.linewidth": 0.6,
})

COMP_PALETTE = [
    "#c0392b", "#2980b9", "#8e44ad", "#27ae60", "#d35400", "#2c3e50",
]
comp_colors = {c["name"]: COMP_PALETTE[i % len(COMP_PALETTE)]
               for i, c in enumerate(COMPARISONS)}


# =============================================================================
# 1. Load DE Results
# =============================================================================

print("\n" + "=" * 70)
print("PATHWAY ANALYSIS AGENT")
print(f"  Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Results from: {RESULTS_DIR}")
print(f"  Output to:    {PATHWAY_DIR}")
print("=" * 70)

print("\nSTEP 1: Loading DE results")

de_results = {}
for comp in COMPARISONS:
    cname = comp["name"]
    cdesc = comp["desc"]
    safe_desc = cdesc.replace(" ", "_").replace("/", "-")

    # Try the naming convention from your pipeline
    pattern = f"DE_{cname}_{safe_desc}.csv"
    fpath = os.path.join(RESULTS_DIR, pattern)

    if not os.path.exists(fpath):
        # Fallback: scan directory for files matching the comparison name
        candidates = [f for f in os.listdir(RESULTS_DIR)
                      if f.startswith(f"DE_{cname}") and f.endswith(".csv")]
        if candidates:
            fpath = os.path.join(RESULTS_DIR, candidates[0])
        else:
            print(f"  ⚠ No DE file found for {cname} — skipping")
            continue

    df = pd.read_csv(fpath)

    # Validate expected columns
    required = {"gene", "log2FC", "pval", "padj"}
    if not required.issubset(df.columns):
        print(f"  ⚠ {cname}: Missing columns {required - set(df.columns)} — skipping")
        continue

    df = df.dropna(subset=["gene", "log2FC", "pval", "padj"])
    df = df.drop_duplicates(subset="gene", keep="first")
    de_results[cname] = df
    n_up = ((df["padj"] < PADJ_THRESH) & (df["log2FC"] > LOG2FC_THRESH)).sum()
    n_dn = ((df["padj"] < PADJ_THRESH) & (df["log2FC"] < -LOG2FC_THRESH)).sum()
    print(f"  ✓ {cname}: {len(df)} genes | {n_up} up, {n_dn} down "
          f"(padj<{PADJ_THRESH}, |log2FC|>{LOG2FC_THRESH})")

if not de_results:
    raise FileNotFoundError(
        f"No DE result files found in {RESULTS_DIR}. "
        "Run the DE pipeline first or check RESULTS_DIR path."
    )


# =============================================================================
# 2. Prepare Gene Set Libraries
# =============================================================================

print("\nSTEP 2: Preparing gene set libraries")


def safe_get_library(name, organism="Mouse"):
    """Attempt to fetch an Enrichr library; return None on failure."""
    try:
        lib = gp.get_library(name=name, organism=organism)
        print(f"  ✓ {name} ({len(lib)} gene sets)")
        return lib
    except Exception as e:
        print(f"  ⚠ {name}: {e}")
        return None


def safe_get_msigdb(category, dbver):
    """Attempt to fetch MSigDB collection; return None on failure."""
    try:
        from gseapy import Msigdb
        msig = Msigdb()
        gmt = msig.get_gmt(category=category, dbver=dbver)
        print(f"  ✓ MSigDB {category} ({len(gmt)} gene sets)")
        return gmt
    except Exception as e:
        print(f"  ⚠ MSigDB {category}: {e}")
        return None


gene_set_dbs = {}

# Enrichr mouse libraries
for lib_name in ENRICHR_MOUSE_LIBS:
    lib = safe_get_library(lib_name, organism="Mouse")
    if lib is not None:
        gene_set_dbs[lib_name] = lib

# MSigDB mouse collections
for label, cat in MSIGDB_MOUSE_COLLECTIONS.items():
    gmt = safe_get_msigdb(cat, MSIGDB_VERSION)
    if gmt is not None:
        gene_set_dbs[f"MSigDB_{label}"] = gmt

if not gene_set_dbs:
    print("\n  ⚠ No gene set libraries loaded. Trying Enrichr online fallback...")
    # Minimal fallback using Enrichr names directly (online mode)
    gene_set_dbs["KEGG_2019_Mouse"] = "KEGG_2019_Mouse"
    gene_set_dbs["GO_Biological_Process_2023"] = "GO_Biological_Process_2023"

print(f"\n  Total databases loaded: {len(gene_set_dbs)}")


# =============================================================================
# Helper: Build ranked gene list
# =============================================================================

def build_rank_metric(df, method=RANK_METRIC):
    """
    Build a ranking metric for GSEA prerank from DE results.

    Methods:
        signed_neg_log10p:  sign(log2FC) × -log10(pval)   [default, recommended]
        log2FC:             raw log2 fold change
        t_statistic:        t-statistic (if available from limma-voom)
    """
    df = df.copy()

    if method == "signed_neg_log10p":
        df["rank_score"] = np.sign(df["log2FC"]) * (-np.log10(df["pval"].clip(lower=1e-300)))
    elif method == "log2FC":
        df["rank_score"] = df["log2FC"]
    elif method == "t_statistic" and "t_statistic" in df.columns:
        df["rank_score"] = df["t_statistic"]
    else:
        df["rank_score"] = np.sign(df["log2FC"]) * (-np.log10(df["pval"].clip(lower=1e-300)))

    rnk = df[["gene", "rank_score"]].dropna()
    rnk = rnk.drop_duplicates(subset="gene", keep="first")
    rnk = rnk.set_index("gene").squeeze().sort_values(ascending=False)
    return rnk


# =============================================================================
# 3. Over-Representation Analysis (ORA)
# =============================================================================

print("\n" + "=" * 70)
print("STEP 3: Over-Representation Analysis (ORA)")
print("=" * 70)

ora_results = {}

for comp in COMPARISONS:
    cname = comp["name"]
    if cname not in de_results:
        continue

    df = de_results[cname]
    sig_up = df[(df["padj"] < PADJ_THRESH) & (df["log2FC"] > LOG2FC_THRESH)]
    sig_dn = df[(df["padj"] < PADJ_THRESH) & (df["log2FC"] < -LOG2FC_THRESH)]

    # Background: all tested genes
    background = df["gene"].tolist()

    ora_results[cname] = {"up": {}, "down": {}}

    for direction, gene_df in [("up", sig_up), ("down", sig_dn)]:
        gene_list = gene_df["gene"].tolist()
        if len(gene_list) < 3:
            print(f"  {cname} {direction}: Only {len(gene_list)} genes — skipping ORA")
            continue

        print(f"\n  {cname} {direction}regulated: {len(gene_list)} genes")

        for db_name, db_val in gene_set_dbs.items():
            try:
                enr = gp.enrich(
                    gene_list=gene_list,
                    gene_sets=db_val,
                    background=background,
                    outdir=None,
                    verbose=False,
                )
                res = enr.results
                if res is not None and not res.empty:
                    sig_terms = res[res["Adjusted P-value"] < ORA_CUTOFF]
                    ora_results[cname][direction][db_name] = res
                    print(f"    {db_name}: {len(sig_terms)} significant terms "
                          f"(of {len(res)} tested)")
                else:
                    print(f"    {db_name}: no results")
            except Exception as e:
                print(f"    {db_name}: error — {e}")

    # Save ORA results
    for direction in ["up", "down"]:
        for db_name, res_df in ora_results[cname][direction].items():
            safe_db = db_name.replace(" ", "_").replace("/", "-")
            out_path = os.path.join(
                PATHWAY_DIR,
                f"ORA_{cname}_{direction}_{safe_db}.csv"
            )
            res_df.to_csv(out_path, index=False)

print("\n  ORA complete.")


# =============================================================================
# 4. GSEA Prerank
# =============================================================================

print("\n" + "=" * 70)
print("STEP 4: GSEA Prerank Analysis")
print("=" * 70)

gsea_results = {}

for comp in COMPARISONS:
    cname = comp["name"]
    if cname not in de_results:
        continue

    df = de_results[cname]
    rnk = build_rank_metric(df, method=RANK_METRIC)
    print(f"\n  {cname}: {len(rnk)} genes ranked by {RANK_METRIC}")

    gsea_results[cname] = {}

    for db_name, db_val in gene_set_dbs.items():
        try:
            pre_res = gp.prerank(
                rnk=rnk,
                gene_sets=db_val,
                threads=GSEA_THREADS,
                min_size=GSEA_MIN_SIZE,
                max_size=GSEA_MAX_SIZE,
                permutation_num=GSEA_PERMUTATIONS,
                outdir=None,
                seed=GSEA_SEED,
                verbose=False,
            )
            res = pre_res.res2d
            if res is not None and not res.empty:
                sig_up = ((res["FDR q-val"] < 0.05) & (res["NES"] > 0)).sum()
                sig_dn = ((res["FDR q-val"] < 0.05) & (res["NES"] < 0)).sum()
                gsea_results[cname][db_name] = {
                    "res2d": res,
                    "prerank_obj": pre_res,
                }
                print(f"    {db_name}: {sig_up} enriched ↑ | {sig_dn} depleted ↓ "
                      f"(FDR<0.05, of {len(res)} sets)")
            else:
                print(f"    {db_name}: no results")
        except Exception as e:
            print(f"    {db_name}: error — {e}")

    # Save GSEA results
    for db_name, gsea_data in gsea_results[cname].items():
        safe_db = db_name.replace(" ", "_").replace("/", "-")
        out_path = os.path.join(
            PATHWAY_DIR,
            f"GSEA_{cname}_{safe_db}.csv"
        )
        gsea_data["res2d"].to_csv(out_path, index=False)

print("\n  GSEA prerank complete.")


# =============================================================================
# 5. Per-Comparison Pathway Figures
# =============================================================================

print("\n" + "=" * 70)
print("STEP 5: Generating per-comparison pathway figures")
print("=" * 70)


def save_fig(fig, name, close=True):
    """Save figure as PNG and PDF."""
    png = os.path.join(PATHWAY_FIG_DIR, f"{name}.png")
    pdf = os.path.join(PATHWAY_FIG_DIR, f"{name}.pdf")
    fig.savefig(png, dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf, bbox_inches="tight", facecolor="white")
    if close:
        plt.close(fig)
    print(f"    Saved: {name}")


# ─── 5a. ORA dot plots ──────────────────────────────────────────────────────

for cname in de_results:
    if cname not in ora_results:
        continue

    for direction in ["up", "down"]:
        for db_name, res_df in ora_results[cname][direction].items():
            sig = res_df[res_df["Adjusted P-value"] < ORA_CUTOFF]
            if sig.empty:
                continue

            try:
                fig, ax = plt.subplots(figsize=(6, max(3, min(len(sig), ORA_TOP_TERMS) * 0.35)))
                top = sig.head(ORA_TOP_TERMS).copy()
                top = top.sort_values("Adjusted P-value", ascending=True)

                # Parse overlap for dot size
                if "Overlap" in top.columns:
                    top["overlap_n"] = top["Overlap"].apply(
                        lambda x: int(str(x).split("/")[0]) if "/" in str(x) else 0
                    )
                else:
                    top["overlap_n"] = 5

                top["neg_log10_padj"] = -np.log10(top["Adjusted P-value"].clip(lower=1e-300))

                scatter = ax.scatter(
                    top["neg_log10_padj"],
                    range(len(top)),
                    s=top["overlap_n"] * 12 + 20,
                    c=top["neg_log10_padj"],
                    cmap="YlOrRd",
                    edgecolors="0.3",
                    linewidths=0.4,
                    zorder=3,
                )
                ax.set_yticks(range(len(top)))
                ax.set_yticklabels(top["Term"].values, fontsize=6)
                ax.set_xlabel("$-$log$_{10}$(adjusted p-value)", fontsize=8)
                ax.axvline(-np.log10(0.05), color="0.5", ls="--", lw=0.6, zorder=1)
                ax.set_title(
                    f"ORA — {cname} ({direction}regulated)\n{db_name}",
                    fontsize=9, fontweight="bold",
                )
                ax.invert_yaxis()
                plt.colorbar(scatter, ax=ax, label="$-$log$_{10}$(padj)", shrink=0.6)

                plt.tight_layout()
                safe_db = db_name.replace(" ", "_").replace("/", "-")
                save_fig(fig, f"ORA_dot_{cname}_{direction}_{safe_db}")

            except Exception as e:
                print(f"    ⚠ ORA plot error ({cname}, {direction}, {db_name}): {e}")


# ─── 5b. GSEA dot plots ─────────────────────────────────────────────────────

for cname in de_results:
    if cname not in gsea_results:
        continue

    for db_name, gsea_data in gsea_results[cname].items():
        res = gsea_data["res2d"]
        sig = res[res["FDR q-val"] < 0.25]  # FDR < 0.25 is standard GSEA reporting
        if sig.empty:
            continue

        try:
            fig, ax = plt.subplots(figsize=(6, max(3, min(len(sig), ORA_TOP_TERMS) * 0.35)))

            top = sig.sort_values("FDR q-val").head(ORA_TOP_TERMS).copy()
            top = top.sort_values("NES", ascending=True)

            colors = ["#2980b9" if x < 0 else "#e74c3c" for x in top["NES"]]
            bars = ax.barh(
                range(len(top)),
                top["NES"].values,
                color=colors, edgecolor="0.3", linewidth=0.4,
            )
            ax.set_yticks(range(len(top)))
            ax.set_yticklabels(top["Term"].values, fontsize=6)
            ax.set_xlabel("Normalized Enrichment Score (NES)", fontsize=8)
            ax.axvline(0, color="0.3", lw=0.6)
            ax.set_title(
                f"GSEA Prerank — {cname}\n{db_name} (FDR<0.25)",
                fontsize=9, fontweight="bold",
            )
            plt.tight_layout()
            safe_db = db_name.replace(" ", "_").replace("/", "-")
            save_fig(fig, f"GSEA_bar_{cname}_{safe_db}")

        except Exception as e:
            print(f"    ⚠ GSEA bar plot error ({cname}, {db_name}): {e}")

    # ─── 5c. GSEA enrichment plots for top terms ────────────────────────────
    for db_name, gsea_data in gsea_results[cname].items():
        pre_obj = gsea_data["prerank_obj"]
        res = gsea_data["res2d"]
        sig = res[res["FDR q-val"] < 0.05].sort_values("NES", ascending=False)

        if len(sig) == 0:
            continue

        # Plot top 3 enriched and top 3 depleted
        top_terms = []
        enriched = sig[sig["NES"] > 0].head(3)["Term"].tolist()
        depleted = sig[sig["NES"] < 0].tail(3)["Term"].tolist()
        top_terms = enriched + depleted

        if len(top_terms) > 0:
            try:
                axs = pre_obj.plot(
                    terms=top_terms[:min(6, len(top_terms))],
                    show_ranking=True,
                    figsize=(4, 5),
                )
                fig = plt.gcf()
                safe_db = db_name.replace(" ", "_").replace("/", "-")
                save_fig(fig, f"GSEA_enrich_{cname}_{safe_db}")
            except Exception as e:
                print(f"    ⚠ GSEA enrichment plot error ({cname}, {db_name}): {e}")


# =============================================================================
# 6. Cross-Comparison Pathway Heatmap
# =============================================================================

print("\n" + "=" * 70)
print("STEP 6: Cross-comparison pathway heatmap")
print("=" * 70)

# Focus on a single key database for the cross-comparison view
# Priority: Hallmark > KEGG > GO_BP
priority_dbs = ["MSigDB_Hallmark", "KEGG_2019_Mouse", "GO_Biological_Process_2023"]
cross_db = None
for pdb in priority_dbs:
    if any(pdb in k for k in gsea_results.get(list(de_results.keys())[0], {})):
        cross_db = [k for k in gsea_results[list(de_results.keys())[0]] if pdb in k][0]
        break

if cross_db is None and gsea_results:
    # Use the first available database
    first_comp = list(gsea_results.keys())[0]
    if gsea_results[first_comp]:
        cross_db = list(gsea_results[first_comp].keys())[0]

if cross_db is not None:
    print(f"  Building cross-comparison heatmap for: {cross_db}")

    # Collect all significant terms across comparisons
    all_terms = set()
    active_comps = []
    for cname in de_results:
        if cname in gsea_results and cross_db in gsea_results[cname]:
            res = gsea_results[cname][cross_db]["res2d"]
            sig = res[res["FDR q-val"] < 0.25]["Term"].tolist()
            all_terms.update(sig)
            active_comps.append(cname)

    if len(all_terms) > 0 and len(active_comps) > 1:
        # Build NES matrix
        nes_matrix = pd.DataFrame(index=sorted(all_terms), columns=active_comps, dtype=float)
        fdr_matrix = pd.DataFrame(index=sorted(all_terms), columns=active_comps, dtype=float)

        for cname in active_comps:
            res = gsea_results[cname][cross_db]["res2d"].set_index("Term")
            for term in nes_matrix.index:
                if term in res.index:
                    nes_matrix.loc[term, cname] = res.loc[term, "NES"]
                    fdr_matrix.loc[term, cname] = res.loc[term, "FDR q-val"]

        nes_matrix = nes_matrix.fillna(0).astype(float)
        fdr_matrix = fdr_matrix.fillna(1.0).astype(float)

        # Filter to top terms (significant in at least one comparison)
        sig_any = (fdr_matrix < 0.25).any(axis=1)
        nes_plot = nes_matrix.loc[sig_any].copy()

        if len(nes_plot) > 40:
            # Keep top 40 by max absolute NES
            max_abs_nes = nes_plot.abs().max(axis=1)
            nes_plot = nes_plot.loc[max_abs_nes.nlargest(40).index]

        if len(nes_plot) > 2:
            fig_h = max(6, len(nes_plot) * 0.35 + 2)
            fig_w = max(6, len(active_comps) * 1.2 + 4)

            fig, ax = plt.subplots(figsize=(fig_w, fig_h))

            # Cluster rows
            from scipy.cluster.hierarchy import linkage, leaves_list
            from scipy.spatial.distance import pdist

            try:
                row_linkage = linkage(pdist(nes_plot.values), method="ward")
                row_order = leaves_list(row_linkage)
                nes_plot = nes_plot.iloc[row_order]
            except Exception:
                pass

            vmax = max(abs(nes_plot.values.min()), abs(nes_plot.values.max()), 1.0)

            im = ax.imshow(
                nes_plot.values, cmap="RdBu_r", aspect="auto",
                vmin=-vmax, vmax=vmax, interpolation="nearest",
            )

            # Significance annotations
            fdr_plot = fdr_matrix.loc[nes_plot.index, active_comps]
            for i in range(len(nes_plot)):
                for j in range(len(active_comps)):
                    fdr = fdr_plot.iloc[i, j]
                    if fdr < 0.001:
                        stars = "***"
                    elif fdr < 0.01:
                        stars = "**"
                    elif fdr < 0.05:
                        stars = "*"
                    elif fdr < 0.25:
                        stars = "·"
                    else:
                        stars = ""
                    if stars:
                        nes_val = nes_plot.iloc[i, j]
                        tc = "white" if abs(nes_val) > vmax * 0.6 else "0.2"
                        ax.text(j, i, stars, ha="center", va="center",
                                fontsize=7, fontweight="bold", color=tc)

            ax.set_xticks(range(len(active_comps)))
            ax.set_xticklabels(active_comps, rotation=45, ha="right", fontsize=7)
            ax.set_yticks(range(len(nes_plot)))
            ax.set_yticklabels(nes_plot.index, fontsize=6)
            ax.tick_params(length=0)

            cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, shrink=0.8)
            cb.set_label("NES", fontsize=8)

            ax.set_title(
                f"GSEA NES Across Comparisons — {cross_db}\n"
                f"(*** FDR<0.001  ** <0.01  * <0.05  · <0.25)",
                fontsize=9, fontweight="bold",
            )
            plt.tight_layout()
            save_fig(fig, f"Cross_comparison_NES_heatmap_{cross_db.replace(' ', '_')}")
        else:
            print("  Too few terms for heatmap.")
    else:
        print("  Insufficient data for cross-comparison heatmap.")
else:
    print("  No GSEA results available for cross-comparison heatmap.")


# =============================================================================
# 7. Leading-Edge Gene Analysis
# =============================================================================

print("\n" + "=" * 70)
print("STEP 7: Leading-edge gene analysis")
print("=" * 70)

leading_edge_all = {}

for cname in de_results:
    if cname not in gsea_results:
        continue

    leading_edge_all[cname] = {}

    for db_name, gsea_data in gsea_results[cname].items():
        res = gsea_data["res2d"]
        sig = res[res["FDR q-val"] < 0.05]

        if sig.empty:
            continue

        le_records = []
        for _, row in sig.iterrows():
            term = row["Term"]
            nes = row["NES"]
            fdr = row["FDR q-val"]
            lead_genes_str = row.get("Lead_genes", "")

            if pd.isna(lead_genes_str) or lead_genes_str == "":
                continue

            genes = [g.strip() for g in str(lead_genes_str).split(";") if g.strip()]
            for g in genes:
                le_records.append({
                    "comparison": cname,
                    "database": db_name,
                    "term": term,
                    "NES": nes,
                    "FDR": fdr,
                    "gene": g,
                })

        if le_records:
            le_df = pd.DataFrame(le_records)
            leading_edge_all[cname][db_name] = le_df

            # Count gene frequency across significant pathways
            gene_freq = le_df["gene"].value_counts()
            n_recurrent = (gene_freq > 1).sum()
            print(f"  {cname} | {db_name}: {len(le_df)} leading-edge entries, "
                  f"{len(gene_freq)} unique genes, {n_recurrent} recurrent")

# Save combined leading-edge table
le_frames = []
for cname, db_dict in leading_edge_all.items():
    for db_name, le_df in db_dict.items():
        le_frames.append(le_df)

if le_frames:
    le_combined = pd.concat(le_frames, ignore_index=True)
    le_combined.to_csv(
        os.path.join(PATHWAY_DIR, "Leading_Edge_Genes_All.csv"), index=False
    )
    print(f"\n  Combined leading-edge table: {len(le_combined)} entries")

    # ─── Recurrent leading-edge gene heatmap ─────────────────────────────
    # Which genes appear as leading edge across multiple comparisons?
    gene_comp_counts = le_combined.groupby(["gene", "comparison"]).size().reset_index(name="n_terms")
    gene_comp_matrix = gene_comp_counts.pivot_table(
        index="gene", columns="comparison", values="n_terms", aggfunc="sum", fill_value=0,
    )

    # Keep genes that appear in ≥2 comparisons
    n_comps_per_gene = (gene_comp_matrix > 0).sum(axis=1)
    recurrent_genes = n_comps_per_gene[n_comps_per_gene >= 2].index.tolist()

    if len(recurrent_genes) > 5:
        top_recurrent = gene_comp_matrix.loc[recurrent_genes].sum(axis=1).nlargest(50).index
        plot_mat = gene_comp_matrix.loc[top_recurrent]

        fig, ax = plt.subplots(figsize=(max(5, len(plot_mat.columns) * 1.2 + 2),
                                         max(4, len(plot_mat) * 0.25 + 1)))
        im = ax.imshow(plot_mat.values, cmap="YlOrRd", aspect="auto", interpolation="nearest")
        ax.set_xticks(range(len(plot_mat.columns)))
        ax.set_xticklabels(plot_mat.columns, rotation=45, ha="right", fontsize=7)
        ax.set_yticks(range(len(plot_mat)))
        ax.set_yticklabels(plot_mat.index, fontsize=6, fontstyle="italic")
        ax.tick_params(length=0)
        cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, shrink=0.7)
        cb.set_label("# Enriched Pathways", fontsize=7)
        ax.set_title("Recurrent Leading-Edge Genes Across Comparisons",
                      fontsize=9, fontweight="bold")
        plt.tight_layout()
        save_fig(fig, "Leading_Edge_Recurrent_Genes_Heatmap")
    else:
        print("  Too few recurrent genes for heatmap.")
else:
    print("  No leading-edge data to summarize.")


# =============================================================================
# 8. Summary Report & Export
# =============================================================================

print("\n" + "=" * 70)
print("STEP 8: Summary Report")
print("=" * 70)

report_lines = []
report_lines.append("=" * 70)
report_lines.append("PATHWAY ANALYSIS SUMMARY REPORT")
report_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.append("=" * 70)
report_lines.append("")
report_lines.append(f"DE Results Directory: {RESULTS_DIR}")
report_lines.append(f"Output Directory:     {PATHWAY_DIR}")
report_lines.append(f"Ranking metric:       {RANK_METRIC}")
report_lines.append(f"ORA thresholds:       padj < {PADJ_THRESH}, |log2FC| > {LOG2FC_THRESH}")
report_lines.append(f"GSEA permutations:    {GSEA_PERMUTATIONS}")
report_lines.append(f"Gene set databases:   {len(gene_set_dbs)}")
for db_name in gene_set_dbs:
    n_sets = len(gene_set_dbs[db_name]) if isinstance(gene_set_dbs[db_name], dict) else "online"
    report_lines.append(f"  • {db_name}: {n_sets} gene sets")
report_lines.append("")

report_lines.append("-" * 70)
report_lines.append("ORA SUMMARY")
report_lines.append("-" * 70)
for cname in de_results:
    if cname not in ora_results:
        continue
    report_lines.append(f"\n  {cname}:")
    for direction in ["up", "down"]:
        for db_name, res_df in ora_results[cname][direction].items():
            n_sig = (res_df["Adjusted P-value"] < ORA_CUTOFF).sum()
            report_lines.append(f"    {direction:>4}  {db_name}: {n_sig} significant terms")

report_lines.append("")
report_lines.append("-" * 70)
report_lines.append("GSEA PRERANK SUMMARY")
report_lines.append("-" * 70)
for cname in de_results:
    if cname not in gsea_results:
        continue
    report_lines.append(f"\n  {cname}:")
    for db_name, gsea_data in gsea_results[cname].items():
        res = gsea_data["res2d"]
        n_up = ((res["FDR q-val"] < 0.05) & (res["NES"] > 0)).sum()
        n_dn = ((res["FDR q-val"] < 0.05) & (res["NES"] < 0)).sum()
        n_up_25 = ((res["FDR q-val"] < 0.25) & (res["NES"] > 0)).sum()
        n_dn_25 = ((res["FDR q-val"] < 0.25) & (res["NES"] < 0)).sum()
        report_lines.append(
            f"    {db_name}:  ↑{n_up} ↓{n_dn} (FDR<0.05) | "
            f"↑{n_up_25} ↓{n_dn_25} (FDR<0.25)"
        )

    # Top enriched / depleted pathways for this comparison
    for db_name, gsea_data in gsea_results[cname].items():
        res = gsea_data["res2d"]
        sig = res[res["FDR q-val"] < 0.05].sort_values("NES", ascending=False)
        if sig.empty:
            continue
        report_lines.append(f"\n    Top enriched — {db_name}:")
        for _, row in sig.head(5).iterrows():
            report_lines.append(f"      NES={row['NES']:+.2f}  FDR={row['FDR q-val']:.1e}  {row['Term']}")
        depleted = sig[sig["NES"] < 0].sort_values("NES")
        if not depleted.empty:
            report_lines.append(f"    Top depleted — {db_name}:")
            for _, row in depleted.head(5).iterrows():
                report_lines.append(f"      NES={row['NES']:+.2f}  FDR={row['FDR q-val']:.1e}  {row['Term']}")

report_lines.append("")
report_lines.append("-" * 70)
report_lines.append("LEADING-EDGE SUMMARY")
report_lines.append("-" * 70)
if le_frames:
    # Most recurrent leading-edge genes overall
    gene_counts = le_combined["gene"].value_counts()
    report_lines.append(f"\n  Total leading-edge entries: {len(le_combined)}")
    report_lines.append(f"  Unique leading-edge genes:  {len(gene_counts)}")
    report_lines.append(f"\n  Top 20 most recurrent leading-edge genes:")
    for g, c in gene_counts.head(20).items():
        # Which comparisons?
        comps_for_gene = le_combined[le_combined["gene"] == g]["comparison"].unique()
        report_lines.append(f"    {g:>15s}  ({c} pathway appearances, "
                            f"comps: {', '.join(comps_for_gene)})")

report_lines.append("")
report_lines.append("=" * 70)
report_lines.append("PATHWAY ANALYSIS COMPLETE")
report_lines.append("=" * 70)

report_text = "\n".join(report_lines)
print(report_text)

# Save report
report_path = os.path.join(PATHWAY_DIR, "Pathway_Analysis_Report.txt")
with open(report_path, "w") as f:
    f.write(report_text)
print(f"\nReport saved: {report_path}")

# ─── Save a master summary CSV ──────────────────────────────────────────────
summary_records = []
for cname in de_results:
    if cname in gsea_results:
        for db_name, gsea_data in gsea_results[cname].items():
            res = gsea_data["res2d"]
            for _, row in res.iterrows():
                summary_records.append({
                    "comparison": cname,
                    "database": db_name,
                    "term": row["Term"],
                    "NES": row.get("NES", np.nan),
                    "NOM_pval": row.get("NOM p-val", np.nan),
                    "FDR_qval": row.get("FDR q-val", np.nan),
                    "lead_genes": row.get("Lead_genes", ""),
                })

if summary_records:
    summary_df = pd.DataFrame(summary_records)
    summary_df.to_csv(
        os.path.join(PATHWAY_DIR, "GSEA_Master_Summary.csv"), index=False
    )
    print(f"Master GSEA summary: {len(summary_df)} rows")

print(f"\n✓ All pathway analysis outputs saved to: {PATHWAY_DIR}/")
print("✓ Figures saved to: {PATHWAY_FIG_DIR}/")
print("Pipeline complete.")



PATHWAY ANALYSIS AGENT
  Timestamp: 2026-07-13 16:08:17
  Results from: results_hippocampus_1pct_min4
  Output to:    pathway_analysis_hippocampus_1pct_min4

STEP 1: Loading DE results
  ✓ AD_vs_Control: 453 genes | 0 up, 0 down (padj<0.05, |log2FC|>0.5)
  ✓ Aged_vs_Young: 453 genes | 3 up, 0 down (padj<0.05, |log2FC|>0.5)
  ✓ YAD_vs_YC: 453 genes | 0 up, 0 down (padj<0.05, |log2FC|>0.5)
  ✓ AAD_vs_AC: 453 genes | 1 up, 8 down (padj<0.05, |log2FC|>0.5)
  ✓ AC_vs_YC: 453 genes | 0 up, 0 down (padj<0.05, |log2FC|>0.5)
  ✓ AAD_vs_YAD: 453 genes | 1 up, 0 down (padj<0.05, |log2FC|>0.5)

STEP 2: Preparing gene set libraries
  ✓ GO_Biological_Process_2023 (5406 gene sets)
  ✓ GO_Molecular_Function_2023 (1147 gene sets)
  ✓ GO_Cellular_Component_2023 (472 gene sets)
  ✓ KEGG_2019_Mouse (303 gene sets)
  ⚠ WikiPathway_2023_Mouse: Sorry. The input: WikiPathway_2023_Mouse could be be found given organism: Mouse
  ✓ Reactome_2022 (1816 gene sets)
  ✓ BioPlanet_2019 (1510 gene sets)
  ⚠ MSigDB mh